# dino.txt 物体槽:2026-11-16 夜间到 11-17 的三组实验

**目的**:(1) 试着提高 dino.txt 的指中率;(2) 为「把 dino.txt 接进现有 VL-JEPA(不蒸馏)」找数据依据。

| 部分 | 实验目录(结果文件在各自的 `results/`) | 问题 |
|---|---|---|
| 1 | `experiments/teacher_hit_rate` | 换打分或换选点方式,能否提高 dino.txt 的指中率 |
| 2 | `experiments/point_probe_offcentre` | 点提示头用偏心点重训,串联误差能降多少 |
| 3 | `experiments/slot_vsibench_check` | 只靠物体槽按规则算答案,VSI-Bench 能答对多少 |

## 总结

- **指中率**:免训练的办法只能从 62.2% 提到 66.7%,有用的只有加一档 448 分辨率。瓶颈在 dino.txt 的分数图本身。每个名字给 16 个候选时,82% 的物体至少有一个候选指中。
- **点提示头重训**:dino.txt 选点时的位置误差从 0.67 m 降到 0.57 m,中心点只损失 0.02 m。象限准确率不变(约 51%),它受指中率限制。
- **物体槽直接算答案**:方向题 65–78%,hardneg_v1 模型 27–51%;相对距离题 44% 对 26%;出现顺序题 30–34%,低于模型的 40.9%。
- **含义**:答方向题和距离题需要的信息能从冻结的部件里取出来,现在的模型缺的是「哪个物体在哪」。第 3 部分的数字是 predictor 完美读取槽时的分数,不是训练后能达到的分数。

## 集成方案(待定,未实现)

- **A 位置槽**:题目里的物体名 → dino.txt 选 patch → 重训过的点提示头(box05)→ 3D 位置。槽 token = 位置编码 + 点提示头特征 + 置信度,经线性层拼在 1024 个场景 token 后面。dino.txt、点提示头、3DRAE 全冻结。
- **B 多候选射线槽**:每个名字 8–16 个候选 patch,槽 token = patch 特征 + 射线 + 分数,不过点提示头。
- **C**:A + B。
- **判据**:real − shuffled,另加「槽打乱」对照(换成别的场景的槽)。

In [1]:
import json
import os
import numpy as np
import pandas as pd
pd.set_option("display.width", 220)

def load(path):
    return json.load(open(path)) if os.path.exists(path) else None

def pct(m, key):
    return f'{100 * m[key]["value"]:.1f}' if key in m else ""

def pct_ci(m, key):
    return f'{100 * m[key]["value"]:.1f} [{100 * m[key]["ci95"][0]:.1f}, {100 * m[key]["ci95"][1]:.1f}]' if key in m else ""

def metres(m, key, ci=False):
    if key not in m:
        return ""
    v = m[key]
    return f'{v["value"]:.2f} [{v["ci95"][0]:.2f}, {v["ci95"][1]:.2f}]' if ci else f'{v["value"]:.2f}'

---

# 第 1 部分 teacher_hit_rate:换打分或换选点方式,能否提高 dino.txt 的指中率

**背景**:串联测试(`slot_chain_check`)里,dino.txt 按名字选出的 patch 有 62% 穿过物体的 3D 框。指中时位置误差 0.41 m,没指中时 1.78 m,所以指中率决定物体槽的质量。这里不训练任何东西,只换打分和选点方式。

**数据**:与串联测试相同。EmbodiedScan 验证集 ScanNet,229 个场景 × 3 个种子,每个样本 8 张带位姿的图,场景内唯一的物体。指中率只统计中心在视野内的物体(4949 个)。

**比较的组合**(脚本 `jepa3d/tools/improve_teacher_grounding.py`)

| 维度 | 取值 |
|---|---|
| 分辨率 | `224`:224×308 的相似度图(原做法);`ms`:再加一张 448×616 的图,池化到同一网格后取平均 |
| 打分 | `centered`:减去该 patch 对全部类别名的平均相似度(原做法);`zscore`:再除以标准差;`softmax`:该名字在全部类别名里的概率(温度 0.02) |
| 选点 | `argmax`:8 个视角里分数最高的 patch(原做法);`smooth3` / `smooth5`:先在每个视角内做 3×3 / 5×5 平均;`epipolar`:每个视角出 2 个候选,用其他视角里靠近这条射线的高分 patch 加分;`consensus`:候选先用点提示头抬到 3D,选周围候选最多的那个;`consensus+`:取获胜候选周围候选位置的加权平均 |

每个被选中的 patch 都再过一遍点提示头得到 3D 位置,所以也有位置误差和方向指标。`224 / centered / argmax` 就是串联测试的做法。方括号是按场景重采样的 95% 置信区间。

In [2]:
R = "../../teacher_hit_rate/results/"
run = load(R + "improve.json")
m = run["metrics"]
SCALES, SCORES = ("224", "ms"), ("centered", "zscore", "softmax")
SELECTS = ("argmax", "smooth3", "smooth5", "epipolar", "consensus", "consensus+")

def grid(key, fmt):
    rows = []
    for scale in SCALES:
        for score in SCORES:
            rows.append({"分辨率": scale, "打分": score, **{s: fmt(m, f"{scale}/{score}/{s}/{key}") for s in SELECTS}})
    return pd.DataFrame(rows)

print("中心在视野内的物体:", m["gt/in_view/position_m"]["n"])
print("原做法 224/centered/argmax 指中率:", pct_ci(m, "224/centered/argmax/in_view/in_box_acc"))

中心在视野内的物体: 4949
原做法 224/centered/argmax 指中率: 62.2 [60.6, 64.6]


## 1.1 指中率(%):所选 patch 的射线穿过物体的 3D 框

`consensus+` 输出的是位置,不是一个 patch,所以没有指中率。

In [3]:
grid("in_view/in_box_acc", pct).drop(columns="consensus+")

,分辨率,打分,argmax,smooth3,smooth5,epipolar,consensus
0,224,centered,62.2,63.8,59.4,63.3,58.6
1,224,zscore,63.2,63.7,59.2,63.4,58.6
2,224,softmax,63.1,65.3,60.9,63.2,63.7
3,ms,centered,65.7,64.5,59.6,65.5,60.7
4,ms,zscore,66.0,64.9,59.6,65.4,59.9
5,ms,softmax,66.7,66.1,61.3,66.6,65.9


## 1.2 位置误差中位数(m),可见物体

对照:类别→位置 probe(训练过)0.95 m;物体中心的真实投影 0.27 m;框内随机 patch 0.47 m。

In [4]:
grid("visible/position_m", metres)

,分辨率,打分,argmax,smooth3,smooth5,epipolar,consensus,consensus+
0,224,centered,0.67,0.66,0.67,0.66,0.72,0.71
1,224,zscore,0.68,0.66,0.67,0.67,0.73,0.72
2,224,softmax,0.66,0.64,0.64,0.66,0.64,0.62
3,ms,centered,0.67,0.66,0.66,0.67,0.72,0.71
4,ms,zscore,0.66,0.65,0.66,0.66,0.72,0.72
5,ms,softmax,0.65,0.64,0.63,0.65,0.64,0.63


## 1.3 象限准确率(%),可见物体

对照:类别→位置 probe 45.6%;真实中心 84.3%;框内随机 patch 74.4%。

In [5]:
grid("visible/quadrant_acc", pct)

,分辨率,打分,argmax,smooth3,smooth5,epipolar,consensus,consensus+
0,224,centered,50.6,50.2,49.5,49.7,46.8,47.0
1,224,zscore,50.4,51.0,49.6,49.3,46.3,46.5
2,224,softmax,50.6,52.2,51.5,50.8,50.2,50.6
3,ms,centered,51.1,50.6,49.5,50.8,47.4,47.7
4,ms,zscore,51.2,51.5,50.3,50.3,46.9,47.2
5,ms,softmax,51.9,52.1,52.4,51.5,51.2,51.5


## 1.4 几个代表方案的完整数字,以及按物体大小拆分的指中率

In [6]:
PICK = ["224/centered/argmax", "224/softmax/smooth3", "ms/centered/argmax", "ms/softmax/argmax", "ms/softmax/smooth3",
        "ms/softmax/consensus", "224/centered/consensus"]
rows = []
for v in PICK:
    rows.append({"方案": v, "指中率": pct_ci(m, f"{v}/in_view/in_box_acc"),
                 "< 0.5 m": pct(m, f"{v}/in_view_size_lt0.5m/in_box_acc"), "0.5–1 m": pct(m, f"{v}/in_view_size_0.5-1m/in_box_acc"),
                 "> 1 m": pct(m, f"{v}/in_view_size_gt1m/in_box_acc"),
                 "位置 (m)": metres(m, f"{v}/visible/position_m", True), "象限": pct_ci(m, f"{v}/visible/quadrant_acc")})
pd.DataFrame(rows)

,方案,指中率,< 0.5 m,0.5–1 m,> 1 m,位置 (m),象限
0,224/centered/argmax,"62.2 [60.6, 64.6]",44.2,61.8,78.0,"0.67 [0.63, 0.73]","50.6 [48.9, 52.6]"
1,224/softmax/smooth3,"65.3 [62.7, 67.6]",47.6,64.1,81.3,"0.64 [0.59, 0.68]","52.2 [50.1, 54.4]"
2,ms/centered/argmax,"65.7 [63.3, 68.0]",50.0,64.1,80.1,"0.67 [0.62, 0.72]","51.1 [49.0, 53.0]"
3,ms/softmax/argmax,"66.7 [64.2, 69.4]",50.8,64.8,81.6,"0.65 [0.60, 0.70]","51.9 [50.0, 53.7]"
4,ms/softmax/smooth3,"66.1 [63.6, 68.4]",48.6,65.2,81.7,"0.64 [0.59, 0.69]","52.1 [49.9, 53.9]"
5,ms/softmax/consensus,"65.9 [63.2, 67.9]",49.2,64.4,81.2,"0.64 [0.60, 0.69]","51.2 [49.3, 53.6]"
6,224/centered/consensus,"58.6 [56.6, 60.9]",40.9,59.6,73.2,"0.72 [0.67, 0.79]","46.8 [45.0, 48.7]"


## 1.5 上限:分数图本身还有多少余地

- `AUC`:穿过框的 patch 分数高于不穿过框的 patch 的概率。
- `候选里有指中的`:每个视角 2 个候选(共 16 个)里至少有一个穿过框。这是 `epipolar` 和 `consensus` 的上限。

In [7]:
pd.DataFrame([{"分辨率": s, "打分": sc, "AUC": f'{m[f"{s}/{sc}/in_view/auc_acc"]["value"]:.3f}',
               "候选里有指中的": pct_ci(m, f"{s}/{sc}/any_cand/in_view/in_box_acc")} for s in SCALES for sc in SCORES])

,分辨率,打分,AUC,候选里有指中的
0,224,centered,0.805,"81.9 [80.0, 83.8]"
1,224,zscore,0.807,"82.7 [80.9, 84.7]"
2,224,softmax,0.807,"82.7 [81.0, 84.4]"
3,ms,centered,0.828,"82.9 [80.8, 84.8]"
4,ms,zscore,0.832,"83.5 [81.5, 85.6]"
5,ms,softmax,0.831,"83.4 [81.5, 85.2]"


## 1 结论

**免训练的办法只能把指中率从 62% 提到 66–67%,提升很小。**

- **有用的只有加一档高分辨率。** `ms` 比 `224` 高 3–4 个点(62.2% → 65.7%),主要来自小物体(< 0.5 m:44% → 50%)。
- **换打分几乎没差别。** `centered`、`zscore`、`softmax` 相差 1 个点以内,在置信区间里。
- **平滑没用。** 3×3 平滑持平,5×5 平滑让小物体掉到 35%(小物体只占一两个 patch,被抹掉了)。
- **跨视角印证没用。** `epipolar` 与 `argmax` 持平;`consensus` 在 `centered` 打分下反而降到 59%。
- **位置误差和方向指标基本不动。** 最好的组合 0.63 m / 52%,原做法 0.67 m / 51%。

**为什么跨视角印证不起作用(推断,未单独验证)**:16 个候选里有指中的占 82–83%,说明对的候选多数时候是有的,但选不出来。场景里唯一的物体往往只在 1–2 个视角里出现,其他视角给不出支持;而错的候选(相似的东西)可能在多个视角都出现。

**含义**
- 指中率的瓶颈在 dino.txt 的分数图本身,不在怎么从图里取点。要再提高,需要训练(比如在分数图上学一个选点头),或者换一种不依赖单点的用法(把多个候选都交给下游)。
- 每个名字给 16 个候选时,82% 的物体至少有一个候选是对的。这支持「一个名字多个候选槽」的集成方式。

**没验证的**:只测了场景内唯一的物体;温度、平滑核、σ 各只试了一个值;一个随机种子(框内随机 patch)。

---

# 第 2 部分 point_probe_offcentre:点提示头用偏心点重训

**背景**:点提示头(输入一个 patch 的 DINOv2 特征和射线,从冻结的 3DRAE token 里读出 3D 位置)训练时只见过物体中心的投影点。dino.txt 选出的 patch 落在物体上任意位置,这时它的误差从 0.27 m 涨到 0.47 m,大物体更明显。

**做法**:重训点提示头(`jepa3d/geo_probe/train.py --box-prompts SHARE`),训练时每个物体以 SHARE 的概率把提示点从中心换成「射线穿过该物体 3D 框的随机 patch」里的随机像素。其余设置与原来相同(958 个训练场景,100 epoch,3DRAE 冻结)。训完用 `improve_teacher_grounding.py` 在同样的验证样本上测。

- `centre`:原来的点提示头(只用中心点训练)
- `box05`:SHARE = 0.5
- `box10`:SHARE = 1.0

注意:框内的 patch 可能被前面的东西挡住,此时 patch 特征是遮挡物的。训练和测试都有这种噪声。

In [8]:
runs = {"centre": load("../../teacher_hit_rate/results/improve.json"), "box05": load("../../point_probe_offcentre/results/box05.json"),
        "box10": load("../../point_probe_offcentre/results/box10.json")}
runs = {k: v["metrics"] for k, v in runs.items() if v}
ROWS = [("gt", "物体中心的真实投影"), ("box", "框内随机 patch"), ("224/centered/argmax", "dino.txt:224 / centered / argmax(原做法)"),
        ("ms/softmax/argmax", "dino.txt:ms / softmax / argmax"), ("ms/softmax/consensus+", "dino.txt:ms / softmax / consensus+")]

def table(key, fmt, **kw):
    return pd.DataFrame([{"提示点来源": label, **{name: fmt(m, f"{v}/{key}", **kw) for name, m in runs.items()}} for v, label in ROWS])

## 2.1 位置误差中位数(m),可见物体

In [9]:
table("visible/position_m", metres, ci=True)

,提示点来源,centre,box05,box10
0,物体中心的真实投影,"0.27 [0.25, 0.29]","0.29 [0.27, 0.31]","0.31 [0.29, 0.33]"
1,框内随机 patch,"0.47 [0.45, 0.49]","0.43 [0.40, 0.45]","0.43 [0.41, 0.45]"
2,dino.txt:224 / centered / argmax(原做法),"0.67 [0.63, 0.73]","0.57 [0.53, 0.62]","0.56 [0.52, 0.61]"
3,dino.txt:ms / softmax / argmax,"0.65 [0.60, 0.70]","0.55 [0.52, 0.59]","0.53 [0.50, 0.58]"
4,dino.txt:ms / softmax / consensus+,"0.63 [0.59, 0.68]","0.53 [0.49, 0.57]","0.52 [0.48, 0.56]"


## 2.2 象限准确率(%),可见物体

In [10]:
table("visible/quadrant_acc", pct_ci)

,提示点来源,centre,box05,box10
0,物体中心的真实投影,"84.3 [82.9, 85.7]","83.2 [81.8, 84.5]","82.4 [81.0, 83.9]"
1,框内随机 patch,"74.4 [73.2, 75.7]","75.6 [74.5, 76.9]","75.5 [74.3, 76.8]"
2,dino.txt:224 / centered / argmax(原做法),"50.6 [48.9, 52.6]","51.3 [49.6, 53.4]","51.7 [49.9, 53.8]"
3,dino.txt:ms / softmax / argmax,"51.9 [50.0, 53.7]","52.8 [50.8, 54.7]","53.0 [51.1, 55.0]"
4,dino.txt:ms / softmax / consensus+,"51.5 [49.7, 53.3]","52.6 [50.6, 54.4]","53.0 [51.1, 55.1]"


## 2.3 按物体大小:框内随机 patch 的位置误差(m),中心在视野内的物体

In [11]:
pd.DataFrame([{"最长边": label, **{name: metres(m, f"box/in_view_size_{g}/position_m") for name, m in runs.items()}}
              for g, label in (("lt0.5m", "< 0.5 m"), ("0.5-1m", "0.5–1 m"), ("gt1m", "> 1 m"))])

,最长边,centre,box05,box10
0,< 0.5 m,0.30,0.30,0.31
1,0.5–1 m,0.41,0.38,0.39
2,> 1 m,0.62,0.56,0.54


## 2.4 dino.txt 指中 / 没指中时的位置误差(m),原做法选点

从逐物体记录里算,中心在视野内的物体。

In [12]:
def hit_miss(path, variant="224/centered/argmax"):
    rows = [json.loads(line) for line in open(path)]
    rows = [r[variant] for r in rows if r["in_view"] and variant in r]
    hit = np.array([r[1] for r in rows]); err = np.array([r[0] for r in rows])
    return {"指中率": f"{100 * hit.mean():.1f}%", "指中时 (m)": f"{np.median(err[hit]):.2f}", "没指中时 (m)": f"{np.median(err[~hit]):.2f}"}

paths = {"centre": "../../teacher_hit_rate/results/improve.objects.jsonl", "box05": "../../point_probe_offcentre/results/box05.objects.jsonl",
         "box10": "../../point_probe_offcentre/results/box10.objects.jsonl"}
pd.DataFrame({k: hit_miss(p) for k, p in paths.items() if os.path.exists(p)}).T

,指中率,指中时 (m),没指中时 (m)
centre,62.2%,0.41,1.78
box05,62.2%,0.36,1.74
box10,62.2%,0.35,1.76


## 2 结论

- **偏心点重训有效**:dino.txt 选点时的位置误差从 0.67 m 降到 0.57 m(box05)/ 0.56 m(box10);框内随机 patch 从 0.47 m 降到 0.43 m,大物体(> 1 m)从 0.62 m 降到 0.54–0.56 m。
- **代价很小**:用真实中心做提示时 0.27 m → 0.29 m(box05)/ 0.31 m(box10)。
- **两个比例差别在置信区间内**。box05 对中心点损失更小,后续用 box05。
- **象限准确率几乎不变**(50.6% → 51–52%):没指中时误差仍是 1.7–1.8 m,重训只改善了指中时的误差(0.41 → 0.36 m)。方向类指标受指中率限制,不受点提示头限制。
- 加上高分辨率和 softmax 打分后最好是 0.52–0.53 m / 53%。

**局限**:验证集 55 个 ScanNet 场景;点提示头没有在 ScanNet++ 上训练或验证过。

---

# 第 3 部分 slot_vsibench_check:只靠物体槽,VSI-Bench 能答对多少

**问题**:如果把「物体槽」接进 predictor,而且 predictor 能完全读懂它,题目能答到什么程度?这里不训练任何东西,直接用槽的位置按题目的规则算答案,和标准答案比。

**做法**(脚本 `jepa3d/tools/slots_answer_vsibench.py`)
1. 从题目里解析出物体名(正则,方向题 3 个名字,相对距离题 4 个选项 + 1 个参照物)。
2. 每个名字:dino.txt 在这道题的 8 张图里选一个 patch(centered 打分),点提示头把它抬成 3D 位置(冻结的 3DRAE token)。
3. 按题目规则算答案:
   - 方向题:站在 A、面向 B,C 在哪边。易:左 / 右;中:左 / 右 / 后(转 ≥ 135° 算后);难:四个象限。
   - 相对距离题:哪个选项的位置离参照物最近(题目按最近点量,这里用槽的位置,即近似物体中心)。
   - 出现顺序题:只用 dino.txt。每个名字取「该视角最高分 ≥ 它最好视角的某个比例」的第一个视角,选与之最一致的选项。

**数据**:VSI-Bench 带位姿的 8 帧版本(`vsibench_test_posed_f8`,1845 题,ScanNet + ScanNet++;去偏子集 811 题)。路径规划题不处理。帧和 hardneg_v1 模型评测时用的完全相同。

**对照**
- `随机 patch`:每个名字随机取一个 patch 再抬成 3D,等于盲猜。
- `hardneg_v1 模型`:目前的 VL-JEPA(`experiments/hardneg_v1`,同一份题和帧)。它的 real − shuffled 只有 +0.5,即基本没用图像。

**注意**:这些场景没有可用的 3D 框(EmbodiedScan 把 ScanNet 验证集放在无标注的测试划分里),所以这里只能和题目答案比,不能检查槽本身指没指中。点提示头只在 ScanNet 上训练过,ScanNet++ 对它是新数据集。

In [13]:
R = "../../slot_vsibench_check/results/"
MODEL = {"test": {"object_rel_direction_easy": 50.8, "object_rel_direction_medium": 29.6, "object_rel_direction_hard": 27.3,
                  "object_rel_distance": 26.2, "obj_appearance_order": 40.9},
         "debiased": {"object_rel_direction_easy": 51.3, "object_rel_direction_medium": 45.9, "object_rel_direction_hard": 23.9,
                      "object_rel_distance": 25.8, "obj_appearance_order": 36.2}}
TYPES = [("object_rel_direction_easy", "方向(易,2 选 1)"), ("object_rel_direction_medium", "方向(中,3 选 1)"),
         ("object_rel_direction_hard", "方向(难,4 选 1)"), ("object_rel_distance", "相对距离(4 选 1)")]
runs = {(probe, split): load(f"{R}{probe}_{split}.json") for probe in ("centre", "box05") for split in ("test", "debiased")}
runs = {k: v["metrics"] for k, v in runs.items() if v}

def table(probe, split, subset="all", sources=("argmax", "smooth3", "consensus")):
    m = runs[probe, split]
    rows = []
    for t, label in TYPES:
        if f"{subset}/{t}/argmax_acc" not in m:
            continue
        rows.append({"题型": label, "题数": m[f"{subset}/{t}/argmax_acc"]["n"], "随机 patch": pct(m, f"{subset}/{t}/random_acc"),
                     "hardneg_v1 模型": MODEL[split][t] if subset == "all" else "",
                     **{f"槽:{s}": pct_ci(m, f"{subset}/{t}/{s}_acc") for s in sources}})
    return pd.DataFrame(rows)

## 3.1 测试集(1845 题里可解析的方向题和相对距离题)

`槽:argmax` 是串联测试的原做法;`smooth3`、`consensus` 是 `teacher_hit_rate` 里的另两种选点方式。点提示头:原来的(只用中心点训练)。方括号是按场景重采样的 95% 置信区间。

In [14]:
table("centre", "test")

,题型,题数,随机 patch,hardneg_v1 模型,槽:argmax,槽:smooth3,槽:consensus
0,"方向(易,2 选 1)",120,45.8,50.8,"77.5 [69.1, 86.0]","76.7 [68.3, 84.9]","85.8 [79.2, 92.6]"
1,"方向(中,3 选 1)",243,43.6,29.6,"70.4 [61.9, 79.1]","70.0 [63.1, 76.3]","65.4 [58.8, 72.5]"
2,"方向(难,4 选 1)",231,25.1,27.3,"64.9 [57.1, 71.7]","67.1 [60.2, 73.8]","57.6 [50.2, 65.7]"
3,相对距离(4 选 1),543,27.6,26.2,"44.4 [39.3, 49.2]","41.8 [37.2, 47.6]","39.8 [33.5, 45.0]"


## 3.2 去偏子集

In [15]:
table("centre", "debiased")

,题型,题数,随机 patch,hardneg_v1 模型,槽:argmax,槽:smooth3,槽:consensus
0,"方向(易,2 选 1)",117,49.6,51.3,"76.9 [68.9, 85.5]","76.1 [68.1, 84.7]","85.5 [78.8, 92.3]"
1,"方向(中,3 选 1)",37,43.2,45.9,"51.4 [33.3, 69.1]","56.8 [38.5, 75.1]","45.9 [30.7, 63.4]"
2,"方向(难,4 选 1)",67,28.4,23.9,"49.3 [33.3, 63.2]","52.2 [40.0, 67.1]","47.8 [32.4, 61.3]"
3,相对距离(4 选 1),209,27.3,25.8,"49.8 [41.5, 57.3]","51.2 [44.3, 58.8]","43.5 [34.7, 51.4]"


## 3.3 按数据集拆分(测试集)

点提示头只在 ScanNet 上训练过。

In [16]:
pd.concat([table("centre", "test", s).assign(数据集=s) for s in ("scannet", "scannetpp")]).drop(columns="hardneg_v1 模型")

,题型,题数,随机 patch,槽:argmax,槽:smooth3,槽:consensus,数据集
0,"方向(易,2 选 1)",63,54.0,"74.6 [63.5, 87.3]","77.8 [66.7, 88.9]","87.3 [77.8, 95.2]",scannet
1,"方向(中,3 选 1)",147,47.6,"79.6 [70.5, 88.4]","78.2 [69.2, 86.8]","73.5 [63.8, 80.9]",scannet
2,"方向(难,4 选 1)",133,23.3,"66.2 [58.6, 75.3]","70.7 [61.8, 78.3]","60.9 [51.5, 69.8]",scannet
3,相对距离(4 选 1),259,27.0,"49.0 [40.0, 58.5]","49.4 [40.9, 59.2]","44.0 [33.9, 53.4]",scannet
0,"方向(易,2 选 1)",57,36.8,"80.7 [68.4, 92.9]","75.4 [58.2, 90.9]","84.2 [73.3, 94.4]",scannetpp
1,"方向(中,3 选 1)",96,37.5,"56.2 [44.3, 69.9]","57.3 [47.3, 68.8]","53.1 [41.4, 64.9]",scannetpp
2,"方向(难,4 选 1)",98,27.6,"63.3 [52.0, 75.8]","62.2 [51.0, 75.8]","53.1 [39.0, 66.4]",scannetpp
3,相对距离(4 选 1),284,28.2,"40.1 [35.1, 45.8]","34.9 [29.8, 39.9]","35.9 [29.4, 41.6]",scannetpp


## 3.4 换成偏心点重训的点提示头(`point_probe_offcentre` 的 box05)

In [17]:
pd.concat([table("box05", s).assign(划分=s) for s in ("test", "debiased")]) if ("box05", "test") in runs else print("没有 box05 的结果")

,题型,题数,随机 patch,hardneg_v1 模型,槽:argmax,槽:smooth3,槽:consensus,划分
0,"方向(易,2 选 1)",120,49.2,50.8,"75.8 [67.2, 84.9]","75.0 [66.9, 84.0]","80.0 [71.7, 88.5]",test
1,"方向(中,3 选 1)",243,44.0,29.6,"74.1 [67.7, 81.4]","72.0 [65.1, 78.9]","67.9 [59.2, 74.7]",test
2,"方向(难,4 选 1)",231,26.8,27.3,"67.1 [59.6, 74.2]","66.2 [58.8, 73.5]","61.5 [55.3, 68.2]",test
3,相对距离(4 选 1),543,28.4,26.2,"44.8 [39.0, 51.0]","43.1 [38.0, 49.2]","44.2 [37.2, 49.7]",test
0,"方向(易,2 选 1)",117,53.8,51.3,"75.2 [66.4, 84.2]","74.4 [64.0, 82.6]","79.5 [70.1, 86.6]",debiased
1,"方向(中,3 选 1)",37,37.8,45.9,"70.3 [50.0, 86.5]","59.5 [38.5, 78.4]","45.9 [30.0, 63.6]",debiased
2,"方向(难,4 选 1)",67,29.9,23.9,"50.7 [33.8, 62.7]","50.7 [37.5, 64.2]","47.8 [34.8, 60.3]",debiased
3,相对距离(4 选 1),209,25.8,25.8,"53.6 [45.3, 61.7]","54.1 [47.0, 61.8]","48.3 [38.9, 57.4]",debiased


## 3.5 dino.txt 的把握和答对率

每道题取各物体「最高分 patch 的标准化分数」里的最小值,即最没把握的那个物体。按这个值把题分成三等份,看 `smooth3` 的准确率(测试集)。

In [18]:
rows = [json.loads(line) for line in open(R + "centre_test.questions.jsonl")]
out = []
for t, label in TYPES:
    r = [x for x in rows if x["type"] == t]
    conf = np.array([min(x["peak_z"]) for x in r]); ok = np.array([x["smooth3"][0] == x["target"].lower() for x in r])
    lo, hi = np.quantile(conf, [1 / 3, 2 / 3])
    out.append({"题型": label, "把握最低的 1/3": f"{100 * ok[conf < lo].mean():.1f}", "中间 1/3": f"{100 * ok[(conf >= lo) & (conf < hi)].mean():.1f}",
                "把握最高的 1/3": f"{100 * ok[conf >= hi].mean():.1f}"})
pd.DataFrame(out)

,题型,把握最低的 1/3,中间 1/3,把握最高的 1/3
0,"方向(易,2 选 1)",60.5,70.7,97.6
1,"方向(中,3 选 1)",59.3,69.1,81.5
2,"方向(难,4 选 1)",59.7,63.6,77.9
3,相对距离(4 选 1),32.4,44.0,48.9


## 3.6 出现顺序题:只用 dino.txt 的逐视角分数

列是「比例」阈值:该视角最高分达到这个名字最好视角的多少,就算它在这一帧出现了。1.0 = 直接取最高分的那一帧。4 选 1,随机 25%。

In [19]:
out = []
for split in ("test", "debiased"):
    m = runs["centre", split]
    for subset in ("all", "scannet", "scannetpp"):
        out.append({"划分": split, "数据集": subset, "题数": m[f"{subset}/obj_appearance_order/order0.8_acc"]["n"],
                    "hardneg_v1 模型": MODEL[split]["obj_appearance_order"] if subset == "all" else "",
                    **{str(s): pct(m, f"{subset}/obj_appearance_order/order{s}_acc") for s in (0.5, 0.6, 0.7, 0.8, 0.9, 1.0)}})
pd.DataFrame(out)

,划分,数据集,题数,hardneg_v1 模型,0.5,0.6,0.7,0.8,0.9,1.0
0,test,all,618,40.9,31.4,33.8,33.7,31.4,30.9,30.1
1,test,scannet,390,,27.4,30.3,30.8,27.9,30.0,30.3
2,test,scannetpp,228,,38.2,39.9,38.6,37.3,32.5,29.8
3,debiased,all,318,36.2,33.0,35.8,34.3,31.8,29.9,28.9
4,debiased,scannet,180,,26.1,30.6,31.1,28.3,28.3,30.0
5,debiased,scannetpp,138,,42.0,42.8,38.4,36.2,31.9,27.5


## 3 结论

- **方向题上,物体槽直接算出的答案比 hardneg_v1 模型高 27–41 个点**(测试集:易 77.5 对 50.8,中 70.4 对 29.6,难 64.9 对 27.3)。模型在这些题上接近随机,而答案所需的信息在槽里。
- **相对距离题提升较小**(44.4 对 26.2):每题 5 个物体都要指对,而且题目按物体最近点量距离,槽给的是中心。
- **出现顺序题不行**(30–34%,模型 40.9):dino.txt 的逐帧分数判断不了物体在哪一帧出现,这类题不该靠槽。
- **dino.txt 的把握和答对率相关**:方向易档,把握最高的三分之一 97.6%,最低的三分之一 60.5%。槽可以带一个置信度。
- **ScanNet++ 上低一些**(方向中档 56.2 对 79.6,距离 40.1 对 49.0),但仍高于随机。
- **选点方法之间没有稳定的优劣**:consensus 在易档最高(85.8)、在中档和难档最低,差别大多在置信区间内。
- **换成 box05 点提示头,差别在置信区间内**(测试集方向中档 74.1 对 70.4,难档 67.1 对 64.9,易档 75.8 对 77.5)。

**局限**

- 这些场景没有可用的 3D 框,只能和题目答案比,不能检查槽本身。
- 这是「predictor 完美读取槽」时的分数,不是训练后能达到的分数。
- 题数少:方向易档 120 题,置信区间约 ±8 个点;去偏子集的方向中档只有 37 题。
- 物体名用正则从题目里取,只覆盖方向、相对距离、出现顺序三类题。